# Voces y documentos peruanos · Extracción

**Fecha de extracción: 25 de octubre de 2026, Perú.**

**Fuentes y licencias:** documentos de Wikimedia Commons y de la Biblioteca Nacional del Perú, con dominio público/CC0/CC BY/CC BY-SA según `datos/seleccion_documentos.csv`; audio OpenSLR 73, Google, CC BY-SA 4.0, y dos entrevistas callejeras de Wikimedia Commons, CC BY-SA 4.0. Fuente oficial: https://www.openslr.org/73/. Cada URL, autor, licencia y procedimiento de recuperación está registrado en los manifiestos y `datos/fuentes.csv`.

**Modelos ejecutados:** PaddleOCR 3.7.0, PP-OCRv6 medium, idioma español, CPU, sin clasificación de orientación ni rectificación; faster-whisper 1.2.1 `small`, CPU/int8, beam 5 y VAD. Diarización mediante WeSpeaker y agrupamiento del módulo `voz.py` de la Sesión 13. Las fotos mantienen sombra, perspectiva y reflejos reales. Se normalizó EXIF y se hicieron recortes de páginas distintas, sin degradación sintética.

**Estado:** resultados de extracción reales. Se completaron veinte referencias manuales y se solicitaron comprobaciones finales en las dos entrevistas. La revisión humana de privacidad y la auditoría continúan en curso. Este notebook no constituye una entrega final mientras esos pasos estén pendientes.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import pandas as pd
from IPython.display import display, Image
import entorno
from pipeline import *
for p in ['paddleocr','paddlepaddle','faster-whisper','pyannote-audio','pandas','plotly']:
    print(p, metadata.version(p))
print('Enunciado: https://github.com/d2cml-ai/Data-Science-Python/issues/190')


## Recuperación e inferencia

Los originales se conservan en `datos/raw/` y se excluyen de Git. `python recuperar_documentos.py` reconstruye las 25 imágenes desde sus URLs, páginas y recortes, verificando SHA256. `python generar_audio_openslr.py` reconstruye 20 montajes de locuciones auténticas; las referencias del proveedor se guardan como metadatos privados y no se usan como referencias manuales. `python agregar_entrevistas.py` recupera las dos entrevistas naturales.

Cambiar `RECALCULAR` a `True` solo para volver a ejecutar los motores. La visualización de las salidas guardadas no requiere volver a inferir.

In [ ]:
RECALCULAR = False
if RECALCULAR:
    extraer_documentos(forzar=True)
    transcribir_audios(forzar=True)
documentos = leer_csv(DOC_CSV)
audios = leer_csv(AUDIO_CSV)
fuentes = leer_csv(DATOS/'fuentes.csv')
print('Documentos:', documentos.shape, '| Audios:', audios.shape, '| Fuentes:', fuentes.shape)
print('Audio total:', round(pd.to_numeric(audios.duracion_seg).sum()/60, 2), 'minutos')
display(documentos.head(5))
display(audios.head(5))


## Limpieza, duplicados y calidad

Se comprueban los archivos reales por SHA256, textos vacíos y duración medida en el WAV. Un OCR corto no implica automáticamente que el original sea ilegible: los manuscritos se inspeccionaron visualmente. El estudiante debe confirmar la legibilidad de la muestra al transcribirla. Los candidatos iniciales que eran retratos, estampillas o imágenes sin la tipología requerida no forman parte del corpus.

In [ ]:
import soundfile as sf
hash_doc = documentos.archivo.map(lambda x: sha256(normalizar_ruta(x)))
hash_audio = audios.archivo_audio.map(lambda x: sha256(normalizar_ruta(x)))
print('Duplicados documentales por hash:', int(hash_doc.duplicated().sum()))
print('Duplicados de audio por hash:', int(hash_audio.duplicated().sum()))
print('Textos OCR vacíos:', int(documentos.texto_ocr.str.strip().eq('').sum()))
print('Transcripciones ASR vacías:', int(audios.texto_asr.str.strip().eq('').sum()))
print('WAV con duración inválida:', sum(sf.info(str(normalizar_ruta(x))).duration <= 0 for x in audios.archivo_audio))
display(leer_csv(DATOS/'descartes.csv'))


## Seudonimización y revisión

El pipeline sustituye patrones de DNI de ocho dígitos, correos y teléfonos. Es una primera pasada: no detecta exhaustivamente nombres ni direcciones. La revisión humana se registra por contenido exacto y se invalida cuando cambian OCR, ASR, referencia, predicción alternativa o JSON. Hasta entonces todos los textos son borradores locales. Los originales y la correspondencia sensible permanecen privados.

In [ ]:
from validar_publicacion import estado
revision = estado()
print('Revisiones humanas completas:', int(revision.revisado.str.lower().isin(['si','sí']).sum()), '/', len(revision))
print('Sustituciones automáticas adicionales necesarias con las reglas básicas:', sum(seudonimizar_basico(t)[1] for t in documentos.texto_ocr) + sum(seudonimizar_basico(t)[1] for t in audios.texto_asr))
print('El recuento anterior no acredita ausencia de nombres: falta revisión humana de todos los textos.')


## Tres gráficos y comprobación de extremos

Los gráficos usan Plotly Express. Los títulos expresan resultados observados y las unidades se mantienen separadas para OCR y ASR. Se adjunta también una imagen estática para la visualización directa en GitHub.

In [ ]:
figuras = analisis_exploratorio()
for i, figura in enumerate(figuras, 1):
    figura.show()
    png = DATOS/'graficos'/f'eda_{i}.png'
    if png.exists(): display(Image(filename=str(png)))


## Hablantes y condiciones de audio

Los 20 montajes de OpenSLR no son entrevistas naturales y pueden contener muchas voces. Los identificadores de procedencia no equivalen a hablantes detectados. Las entrevistas callejeras añaden ruido y cambios espontáneos de turno. El agrupamiento puede fusionar voces o dividir una sola; el número detectado requiere contraste auditivo.

In [ ]:
meta_audio = leer_csv(DATOS/'fuentes_audio.csv')
display(audios[['id','duracion_seg','n_hablantes','modelo_asr']])
display(meta_audio[['id','fuente_url','procedencia','n_voces_fuente']])
print('Corpus final extraído:', len(documentos), 'documentos y', len(audios), 'clips.')
print('Pendientes para entregar: comprobación final de referencias, auditoría, privacidad, evidencias y despliegue público.')
